# 03 · F0 モデル（PrettyPitch）の使い方

PrettyPitch は F0 を **3 チャンネル**で生成します。

```
F0 = 2^( baseline + plain + vibrato )
      │           │       └ ビブラート（4.5–7.5 Hz の周期的なゆれ）
      │           └ plain = 抑揚（しゃくり・落ち・うねり）
      └ baseline = 楽譜の音高を階段状にしてガウシアン平滑（σ20 ms）したもの
```

**plain と vibrato が別チャンネルなので、抑揚とビブラートを独立に増減できます。**
さらに 2 段ピッチ補正は **plain にしか掛かりません**。だからビブラートを強くしても
音程が目標からずれません（形が保たれる）。

音声で確かめるセルがあるので、notebook 01 と同じ準備（LeapSinger の音響ウェイトを
`checkpoints/leapsinger/` に、LeapSinger のリポジトリを隣に）が要ります。F0 のプロットだけなら
`render_with_f0` を呼ぶセルを飛ばしてください。

扱う内容:

1. `plain_scale` / `vib_scale` による全体の強度変更
2. 両者が本当に独立であることの検証
3. フレーム別の配列で、フレーズの中で強度を時間変化させる
4. `style_mix` による連続的なスタイル混合
5. 落とし穴

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np, torch
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from svs.render import load_models, render_phrase, _f0_score_from_seq
from svs.score_io import load_score
from svs.duration_postprocess import postprocess_durations
from svs.plot import plot_f0

SPK_ID, SEED = 2, 0
ctx = load_models(spk_id=SPK_ID, device="cpu")

phrases = load_score(ROOT / "examples" / "kirakiraboshi.ust")
phr = phrases[0]
seq = postprocess_durations(phr, rest_before=1e9)
score = _f0_score_from_seq(seq, spk_id=SPK_ID, style_id=0)

fr = ctx.frame_rate
print()
print("フレーズ0:", " ".join(seq.phonemes))
print(f"{len(score['phonemes'])} スロット / {seq.dur_sec.sum():.2f} 秒")

## 1. `plain_scale`（抑揚）と `vib_scale`（ビブラート）

`synthesize_f0` の引数です。内部ではこう使われています。

```python
plain_delta = out3[0] * plain_std * plain_scale     # 抑揚
vib_delta   = out3[1] * vib_std   * vib_scale       # ビブラート
...
plain_hz = 2**(baseline + plain_delta)
plain_hz = 補正(plain_hz)                           # ← 補正は plain にだけ
f0_hz    = plain_hz * 2**vib_delta                  # ← vibrato は最後に掛け戻す
```

In [ ]:
def synth(plain_scale=1.0, vib_scale=1.0, **kw):
    return ctx.pp.synthesize_f0(ctx.f0_model, ctx.f0_cfg, score, device="cpu", seed=SEED,
                                plain_scale=plain_scale, vib_scale=vib_scale, **kw)

def to_midi(o):
    return np.where(o["uv"] > 0, 69 + 12*np.log2(np.maximum(o["f0_hz"], 1e-6)/440), np.nan)

base = synth()
t = np.arange(len(base["f0_hz"])) / fr

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
for s in (0.0, 0.5, 1.0, 2.0):
    o = synth(plain_scale=s)
    axes[0].plot(t, to_midi(o), lw=1.1, label=f"plain_scale={s}")
    axes[1].plot(t, o["plain_delta"]*1200, lw=1.0, label=f"plain_scale={s}")
axes[0].plot(t, 69 + 12*(base["baseline_log2"] - np.log2(440)), "--", color="#888", lw=1, label="baseline")
axes[0].set_ylabel("pitch [MIDI]"); axes[0].legend(fontsize=8)
axes[0].set_title("plain_scale sweep (vib_scale=1)")
axes[1].set_ylabel("plain [cent]"); axes[1].set_xlabel("time [s]"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
for s in (0.0, 1.0, 2.0, 3.0):
    o = synth(vib_scale=s)
    axes[0].plot(t, to_midi(o), lw=1.1, label=f"vib_scale={s}")
    axes[1].plot(t, o["vib_delta"]*1200, lw=1.0, label=f"vib_scale={s}")
axes[0].set_ylabel("pitch [MIDI]"); axes[0].legend(fontsize=8)
axes[0].set_title("vib_scale sweep (plain_scale=1)")
axes[1].set_ylabel("vibrato [cent]"); axes[1].set_xlabel("time [s]"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

for s in (0.0, 1.0, 2.5):
    print(f"vib_scale={s}")
    display(Audio(render_phrase(ctx, phr, seed=SEED, vib_scale=s)["wav"], rate=ctx.sample_rate))

## 2. 独立性の検証

同じ seed なら、`plain_scale` を変えても `vib_delta` は **1 ビットも変わりません**
（逆も同様）。モデルは同じ `out3` を出し、2 つのチャンネルを別々にスケールしているだけだからです。

さらに「**ビブラートを強めても音程は崩れない**」ことも確かめます。ここは指標の取り方に注意が要ります。
ビブラートは定義上ゆれなので、`vib_scale` を上げれば**瞬時の**偏差は当然大きくなります。
見るべきは「**ノートの平均音高が目標に乗ったままか**」です。
補正が plain にしか掛からない以上、vibrato は平均 0 のゆれを足しているだけなので、
平均は動かないはずです。


In [ ]:
a, b = synth(plain_scale=1.0), synth(plain_scale=3.0)
print("plain_scale を変えても vib_delta は不変 :", np.array_equal(a["vib_delta"], b["vib_delta"]))
c, d = synth(vib_scale=1.0), synth(vib_scale=3.0)
print("vib_scale を変えても plain_delta は不変 :", np.array_equal(c["plain_delta"], d["plain_delta"]))

# ノート区間と「ノートの公称音高」を取る（2 段補正が使っているのと同じ区切り方）
from prettypitch.baseline import phoneme_frame_bounds
from prettypitch.correction import build_per_frame_arrays

slots = score["phonemes"]
ph_strs = [s["ph"] for s in slots]
dur_sec = np.array([float(s["dur"]) for s in slots], np.float32)
values = np.array([float(s["midi"]) if "midi" in s else np.nan for s in slots])
T0 = len(base["f0_hz"])
fb0 = phoneme_frame_bounds(dur_sec, fr, T0)
pd0 = np.maximum(np.diff(fb0), 1).astype(np.int64)
ph_pf, note_pitch_pf, nb_pf = build_per_frame_arrays(ph_strs, pd0, values, T0)
nuc = frozenset(ctx.f0_cfg["nucleus"])
is_vow_pf = np.array([p in nuc for p in ph_pf])
starts0 = np.where(nb_pf)[0]
segs0 = list(zip(starts0, np.append(starts0[1:], T0)))

def deviation(o):
    # 母音フレームで「生成F0 - ノート公称音高」[cent] を取り、
    # (ノート平均の絶対値の平均, 瞬時偏差の中央値) を返す
    v = (o["uv"] > 0) & is_vow_pf & np.isfinite(note_pitch_pf)
    cur = 69 + 12*np.log2(np.maximum(o["f0_hz"], 1e-6)/440)
    dev = (cur - note_pitch_pf) * 100
    per_note = [np.mean(dev[s:e][v[s:e]]) for s, e in segs0 if v[s:e].any()]
    return float(np.mean(np.abs(per_note))), float(np.median(np.abs(dev[v])))

print()
print(f'{"vib_scale":>10} | {"ノート平均のずれ":>16} | {"瞬時のゆれ幅(中央値)":>20}')
print("-" * 56)
for s in (0.0, 1.0, 2.0, 3.0):
    m, i = deviation(synth(vib_scale=s))
    print(f"{s:>10} | {m:>13.1f} cent | {i:>17.1f} cent")
print()
print("ノート平均は vib_scale に依らずほぼ一定（音程は崩れていない）。")
print("増えているのは瞬時のゆれ幅＝ビブラートそのもの。")


## 3. 強度を時間で変える

`plain_scale` / `vib_scale` はスカラーだけでなく **フレーム別の配列**も受けます。
長さ T の配列を渡せば、フレーズの途中で強さを変えられます。

ここではビブラートを曲の頭から終わりへ向けて **0 → 2 へ連続的に上げて**みます。
配列を作るだけなので、ノートや音素の区切りを数える必要はありません。


### 3-1. まず健全性の確認

ゲインを全部 1.0 にすれば、`synthesize_f0`（既定引数）と一致するはずです。


In [ ]:
ref = ctx.pp.synthesize_f0(ctx.f0_model, ctx.f0_cfg, score, device="cpu", seed=SEED)
Tm = len(ref["f0_hz"])

flat = ctx.pp.synthesize_f0(ctx.f0_model, ctx.f0_cfg, score, device="cpu", seed=SEED,
                            vib_scale=np.ones(Tm))
print("ゲイン一定 1.0 で synthesize_f0 と一致:",
      np.allclose(ref["f0_hz"], flat["f0_hz"], atol=1e-4),
      f"  max|d| = {np.abs(ref['f0_hz'] - flat['f0_hz']).max():.2e} Hz")
print(f"フレーム数 T = {Tm}  ({Tm / fr:.2f} 秒)")


### 3-2. だんだん強くする

頭では平坦に、終わりでは大きく揺れるようにします。


In [ ]:
vib_gain = np.linspace(0.0, 2.0, Tm)          # 長さ T の配列を渡すだけ
o_ramp = ctx.pp.synthesize_f0(ctx.f0_model, ctx.f0_cfg, score, device="cpu", seed=SEED,
                              vib_scale=vib_gain)

tt_ = np.arange(Tm) / fr
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
axes[0].plot(tt_, vib_gain, lw=1.8, color="#1565c0")
axes[0].set_ylabel("vib gain"); axes[0].set_title("vib_scale as a per-frame array (0 -> 2)")
axes[1].plot(tt_, ref["vib_delta"] * 1200, lw=0.9, color="#bbb", label="vib_scale=1.0")
axes[1].plot(tt_, o_ramp["vib_delta"] * 1200, lw=0.9, color="#1565c0", label="ramp 0 -> 2")
axes[1].set_ylabel("vibrato [cent]"); axes[1].legend(fontsize=9)
axes[2].plot(tt_, to_midi(ref), lw=1.0, color="#bbb", label="vib_scale=1.0")
axes[2].plot(tt_, to_midi(o_ramp), lw=1.1, color="#1565c0", label="ramp 0 -> 2")
axes[2].set_ylabel("pitch [MIDI]"); axes[2].set_xlabel("time [s]"); axes[2].legend(fontsize=9)
plt.tight_layout(); plt.show()

q = Tm // 4
print("ビブラート振幅の最大値 [cent]")
print(f"{'区間':>10} {'vib_scale=1.0':>14} {'ramp 0->2':>11} {'平均ゲイン':>10}")
for i in range(4):
    s, e = i * q, (i + 1) * q
    print(f"{i*25:>3}-{(i+1)*25:>3}%  {np.abs(ref['vib_delta'][s:e]).max()*1200:14.1f}"
          f" {np.abs(o_ramp['vib_delta'][s:e]).max()*1200:11.1f} {vib_gain[s:e].mean():10.2f}")
print(f"\nplain は無傷: max|d| = "
      f"{np.abs(o_ramp['plain_delta'] - ref['plain_delta']).max()*1200:.3f} cent")


### 3-3. 音で確かめる

F0 だけ差し替えて、音響モデル以降を回します。


In [ ]:
from svs.duration_postprocess import to_frames
from svs.render import _align_durs

def render_with_f0(f0o, style_id=0, num_steps=None):
    # 既に作った F0 を使って mel -> wav まで回す（render_phrase の後半だけ）
    if num_steps is None:
        num_steps = int(ctx.ac_cfg.get("infer_steps", 1))
    f0_hz = np.asarray(f0o["f0_hz"], np.float32); uv = np.asarray(f0o["uv"], np.float32)
    ph_durs, _ = to_frames(seq.dur_sec, ctx.frame_rate)
    ph_durs = _align_durs(ph_durs, len(f0_hz))
    logf0 = np.log2(np.maximum(f0_hz, 1.0)).astype(np.float32)
    ph_ids = np.asarray([ctx.ph2id[p] for p in seq.phonemes], np.int64)
    if ctx.use_uv:
        item = dict(ph_ids=ph_ids, ph_durs=ph_durs, f0_logf0=logf0, uv=uv,
                    spk_id=ctx.spk_id, style_id=style_id)
        mel = ctx.ls.infer_mel(ctx.ac_model, item, num_steps=num_steps, device="cpu", seed=SEED)
    else:
        mel = ctx.ls.infer_mel_uvfree(ctx.ac_model, ph_ids, ph_durs, f0_hz,
                                      num_steps=num_steps, spk_embed=ctx.spk_vec,
                                      device="cpu", seed=SEED)
    return np.asarray(ctx.ls.mel_to_wav(ctx.voc, mel, logf0, uv), np.float32)

print("ビブラートを 0 から 2 へ上げていく")
display(Audio(render_with_f0(o_ramp), rate=ctx.sample_rate))
print("全体一律 vib_scale=1.0（比較）")
display(Audio(render_with_f0(ref), rate=ctx.sample_rate))


## 4. `style_mix`：スタイルの連続混合

このモデルは `n_styles=3`（0 = 既定 / 1 = ritsu normal / 2 = ritsu soft）。
`synthesize_f0` は離散の `style_id` も受けますが、**モデル自体は連続混合に対応**しています
（`style_mix @ style_emb.weight`）。one-hot を渡せば `style_id` と出力が完全に一致します。

**ただし F0 に出る差は小さく、聴き分けはできません。** 既定と soft の差は中央値 3.5 cent、
6 割のフレームが 5 cent 未満です。理由は、3 つの音源の違いが**声色**にあり、音高の振る舞いの
差はもともと小さいためです。学習データで測ると、ビブラートの大きさはほぼ同じで、抑揚の
大きさだけが kire → normal → soft と 13〜17% 小さくなります。声色を担うのは音響モデルで、
このノートブックが使う同梱ウェイトはスタイルを受け取りません。

学習されていないわけではありません。評価句 112 件で正しいスタイルと誤ったスタイルを与えて
比べると、**3 スタイルすべてで正しいものが最小誤差**になります（改善幅 0.4〜2.0%）。

| 真のスタイル | style0 を与える | style1 | style2 |
|---|---|---|---|
| 0 kire | **0.1786** | 0.1811 | 0.1821 |
| 1 normal | 0.1598 | **0.1586** | 0.1592 |
| 2 soft | 0.1537 | 0.1522 | **0.1513** |

下のプロットで線が重なって見えるのは、この結論と一致しています。


In [ ]:
def synth_style_mix(weights, seed=SEED):
    # weights: 長さ n_styles の混合比。one-hot なら style_id と同じ結果。
    return ctx.pp.synthesize_f0(ctx.f0_model, ctx.f0_cfg, score, device="cpu", seed=seed,
                                style_mix=np.asarray(weights, np.float32))

base_style = synth_style_mix([1.0, 0.0, 0.0])

fig, ax = plt.subplots(figsize=(14, 4))
for w in np.linspace(0, 1, 5):
    o = synth_style_mix([1.0 - w, 0.0, w])
    ax.plot(np.arange(len(o["f0_hz"])) / fr, to_midi(o), lw=1.0,
            label=f"default {1-w:.2f} / soft {w:.2f}")
ax.set_ylabel("pitch [MIDI]"); ax.set_xlabel("time [s]")
ax.set_title("continuous style_mix (default -> soft) — the lines overlap, and that is the point")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print("既定との差（有声フレームの音高）")
for w in np.linspace(0, 1, 5):
    o = synth_style_mix([1.0 - w, 0.0, w])
    m = (base_style["uv"] > 0) & (o["uv"] > 0)
    d = np.abs((np.log2(o["f0_hz"][m]) - np.log2(base_style["f0_hz"][m])) * 1200)
    print(f"  soft {w:.2f}  平均 {d.mean():5.1f} cent  中央値 {np.median(d):5.1f}"
          f"  5 cent 未満 {(d < 5).mean()*100:3.0f}%")

print("\n聴いて確かめる（差は分からないはずです）")
for w in (0.0, 1.0):
    print(f"style_mix = [{1-w:.1f}, 0.0, {w:.1f}]  (default / normal / soft)")
    display(Audio(render_with_f0(synth_style_mix([1.0 - w, 0.0, w])), rate=ctx.sample_rate))


## 5. 落とし穴

* **seed を固定する。** 拡散モデルなので、seed が違えば抑揚もビブラートも変わります。
  比較するときは必ず同じ seed で。
* **`plain_scale` を上げすぎると 2 段補正が打ち消す。** 補正は「ノートの知覚音高を
  楽譜の音高に合わせる」ので、抑揚を極端に強めても平均は目標へ引き戻されます。
  抑揚そのものを見たいときは `correction=False`・`hann_alpha=0` で比べてください。
* **pau は常に無声。** `uv` は `(vuv > 0.5) & ~pau` なので、pau フレームは必ず無声になります。
* **`style_mix` は `style_id` より優先される。** 両方渡すと `style_mix` が勝ちます
  （実装が `elif` のため）。
* **音響モデルのスタイルとは別物。** ここで扱っているのは F0 モデル（と子音長モデル）のスタイルです。
  既定の音響モデル（v/uv フリー）にスタイルはありません。